#### Environment Setup & API Keys

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:


from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

C:\Users\impav\AppData\Local\Temp\ipykernel_8596\253967555.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\Ai Engineer_\Week2\day2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3, api_key=os.getenv("GROQ_API_KEY"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2057.18it/s]


#### PDF Document Loading

In [5]:
file_path="data/SDG.pdf"
loder= PyPDFLoader(file_path)
data=loder.load()


In [ ]:
data

#### Text Extraction for Question Generation

In [7]:
question_gen=""
for page in data:
  question_gen+=page.page_content

In [8]:
question_gen

'IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 yea

#### Token Splitting (Large Chunks for Questions)

In [9]:
splitter_ques_gen=TokenTextSplitter(
  model_name="gpt-3.5-turbo",
  chunk_size=10000,
  chunk_overlap=200
)

In [10]:
chunk_question_gen=splitter_ques_gen.split_text(question_gen)

In [11]:
chunk_question_gen

['IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 ye

In [12]:
type(chunk_question_gen[0])

str

In [13]:
from langchain_core.documents import Document


#### String Chunks to Document Objects Conversion

In [14]:
document_ques_gen=[Document(page_content=t) for t in chunk_question_gen]
document_ques_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

In [15]:
type(document_ques_gen[0])

langchain_core.documents.base.Document

In [16]:
splitter_ans_gen=TokenTextSplitter(
  model_name="gpt-3.5-turbo",
  chunk_size=1000,
  chunk_overlap=100
)

#### Smaller Chunks for Answer Generation VectorStore

In [17]:
document_ans_gen=splitter_ans_gen.split_documents(
  document_ques_gen
)

In [18]:
document_ans_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

In [19]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3, api_key=os.getenv("GROQ_API_KEY"))
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4209.30it/s]


In [ ]:
prompt_template = """
You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

------------
{text}
------------

Create only 10 questions that will prepare the coders or programmers
Make sure not to lose any important information.

QUESTIONS:
"""

In [21]:
prompt_questions=PromptTemplate(template=prompt_template,input_variables=['text'])

In [22]:
refine_template = ("""
You are an expert at creating practice questions based on coding material and documentation.
Your goal is to help a coder or programmer prepare for a coding test.
We have received some practice questions to a certain extent: {existing_answer}.
We have the option to refine the existing questions or add new ones
(only if necessary) with some more context below.
----------
{text}
----------

Given the new context, refine the original questions in English.
If the context is not helpful, please provide the original questions.
QUESTIONS:
"""
)


In [23]:
refine_prompt_questions=PromptTemplate(template=refine_template,input_variables=["existing_answer","text"])

In [24]:
from langchain_classic.chains.summarize import load_summarize_chain

In [25]:
ques_gen_chain=load_summarize_chain(llm=llm,chain_type="refine",verbose=True,question_prompt=prompt_questions,refine_prompt=refine_prompt_questions)

In [26]:
ques=ques_gen_chain.run(document_ques_gen)
print(ques)

C:\Users\impav\AppData\Local\Temp\ipykernel_8596\137075838.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  ques=ques_gen_chain.run(document_ques_gen)




> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

------------
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They knew we had enough 
food to feed the world, but that it wasn’t getting shared. They knew there 
were medicines for HIV and other diseases, but they cost a lot. They knew 
that earthquakes and floods were inevitable, but that the high death 
tolls were not. 
They also knew that billions of people worldwide shared their hope for a 
better future.
So leaders from

In [27]:
vectorstore=FAISS.from_documents(document_ans_gen,embeddings)

In [28]:
ques

"Here are some questions based on the text to prepare coders or programmers for their exam and coding tests:\n\n**Section 1: Introduction to Sustainable Development Goals (SDGs)**\n\n1. What year did leaders from 193 countries come together to create the Sustainable Development Goals (SDGs)?\n2. What are the three main problems mentioned in the text that the SDGs aim to address?\n3. What is the target year for achieving the SDGs?\n\n**Section 2: SDG 1 - End Extreme Poverty**\n\n1. What is the current number of people living in extreme poverty worldwide?\n2. What was the goal set in 2000 regarding extreme poverty, and was it achieved?\n3. What is the new goal for ending poverty, and by what year?\n\n**Section 3: SDG 2 - End Hunger**\n\n1. What progress has been made in reducing hunger over the past 20 years?\n2. What percentage of the global population still goes to bed hungry every night?\n3. What are some ways to promote sustainable agriculture and support small farmers?\n\n**Section 

In [29]:
import re
questions_list = re.findall(r"^\d+\.\s*(.*)", ques, flags=re.MULTILINE)

In [30]:
questions_list

['What year did leaders from 193 countries come together to create the Sustainable Development Goals (SDGs)?',
 'What are the three main problems mentioned in the text that the SDGs aim to address?',
 'What is the target year for achieving the SDGs?',
 'What is the current number of people living in extreme poverty worldwide?',
 'What was the goal set in 2000 regarding extreme poverty, and was it achieved?',
 'What is the new goal for ending poverty, and by what year?',
 'What progress has been made in reducing hunger over the past 20 years?',
 'What percentage of the global population still goes to bed hungry every night?',
 'What are some ways to promote sustainable agriculture and support small farmers?',
 'What is the current state of healthcare worldwide, and what are some of the challenges?',
 'What progress has been made in reducing preventable child deaths and maternal mortality?',
 'What are some ways to ensure everyone has access to safe and effective medicines and vaccines?'

In [31]:
# answer_generation_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="stuff",
#     retriever=vectorstore.as_retriever()
# )

In [32]:
# LCEL
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":4,
        "fetch_k":10
    }
)

template = """
Answer the question only from the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    RunnableParallel({
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    })
    | prompt
    | llm
    | StrOutputParser()
)

In [33]:
for question in questions_list:
  print("Question: ",question)
  # ans=answer_generation_chain.run(question)
  ans = chain.invoke(question)
  print("Answer: ",ans)
  print("********")

  with open("answers.txt","a") as f:
    f.write("Question: "+ question + "\\n")
    f.write("**********")

Question:  What year did leaders from 193 countries come together to create the Sustainable Development Goals (SDGs)?
Answer:  2015
********
Question:  What are the three main problems mentioned in the text that the SDGs aim to address?
Answer:  The three main problems mentioned in the text that the SDGs aim to address are:

1. Poverty and hunger: The text states that there are still over 800 million people living in extreme poverty and nearly 1 out of every 9 people on earth going to bed hungry every night.
2. Climate change: The text mentions that climate change is causing drastic effects, including water scarcity, loss of life and property, and making the oceans more acidic.
3. Inequality: The text states that there are gross inequalities in work and wages, and that women and girls lag behind in many areas, including education, employment, and public decision-making.
********
Question:  What is the target year for achieving the SDGs?
Answer:  2030
********
Question:  What is the cur

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kwywd67sf5gvfjzphxn3bfp2` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 96717, Requested 4670. Please try again in 19m58.368s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}